In [ ]:
def compute_extended_fitness_with_imports(
    Mcp_export: pd.DataFrame,
    Mcp_import: pd.DataFrame,
    C: pd.DataFrame,
    alpha: float,
    tensor_window: int = 1,
    max_iter: int = 1000,
    tol: float = 1e-12,
    track_convergence: bool = True,
    save_delta: bool = False,
    delta_window: int | None = None,
):
    """
    Fitness estesa con import/export e matrice di composizione C.

    Input:
    - alpha: 
    - tensor_window: profondita' del tensore usato nel calcolo della penalita'
      da input importati (ordini di distanza in C). 
    - delta_window 
    - track_convergence
    - save_delta

    """

    # 0) Gestione INPUT

    # 0.1) Matrici
    M_export = Mcp_export.values.astype(float)
    M_import = Mcp_import.values.astype(float)
    C = C.values.astype(float)

    n_c, n_p = Mcp_export.shape

    # 1) Creazione Tensori

    # 1.a) Matrice di propagazione B.
    # La penalita' e' un prodotto vettore-riga per matrice,
    #     sum_p' (Delta ⊙ M^I)_{c p'} (C^m)_{p' p},
    # mentre l'aggiornamento di Q e' un prodotto matrice-vettore, C @ Q: le due
    # contrazioni percorrono la matrice in direzioni opposte. Poiche' a questa
    # funzione si passa C gia' trasposta, la matrice da usare a destra nella
    # penalita' e' B = C.T, cioe' la C originale come letta dal file.
    # Le potenze di C NON vengono mai materializzate: lo schema di Horner
    # (piu' sotto) moltiplica per B un passo alla volta. Si evitano cosi' un
    # tensore (tensor_window, n_p, n_p) - 2 GB a HS6 con finestra 10 - il
    # relativo precalcolo, e la perdita di precisione dovuta a entrate ~rho^k.
    B = sp.csr_matrix(np.ascontiguousarray(C.T))

    # 2) Algoritmo

    # 2.1) Inizializzazione
    F     = np.ones(n_c)
    Q     = np.ones(n_p)
    # una slice in piu': C^(k+1) va accoppiato con Delta[:, :, k+1],
    # che ha accumulato k+2 normalizzazioni (la teoria vuole C^m con m+1)
    Delta = np.zeros((tensor_window + 1, n_c, n_p))

    # *) SALVATAGGI OPZIONALI

    # *.a) CONVERGENZA
    history_convergence = [] if track_convergence else None
    # *.b) DELTA
    delta_hist = deque(maxlen=delta_window) if save_delta else None

    # 2.2) Sistema dinamico
    i = 0
    for i in range(max_iter):
        F_old = F.copy()
        Q_old = Q.copy()

        # 2.2.1) Fitness F

        # 2.2.1.a) Calcolo import penalty (schema di Horner)
        #
        #   S = sum_(m=1..K) alpha^m (Delta[m] * M_import) B^m
        #     = alpha ( X_1 + alpha ( X_2 + ... + alpha X_K B ) B ) B
        #
        # con X_m = Delta[m] * M_import (prodotto elemento-per-elemento).
        # Ogni passo e' una sola moltiplicazione per B: nessuna potenza viene
        # costruita. Delta[m] porta gia' le m+1 normalizzazioni per mean(Q)
        # che competono all'ordine m.
        K = tensor_window

        # FILTRO 1 (import): applicato a ogni ordine separatamente
        R = Delta[K] * M_import                      # X_K
        for m in range(K - 1, 0, -1):
            R = R @ B                                # un passo a valle nella rete
            R = Delta[m] * M_import + alpha * R      # aggiungo l'import di ordine m
        S = alpha * np.asarray(R @ B)                # (n_c, n_p)

        # FILTRO 2 (export): identico per tutti gli ordini, quindi si applica
        # una volta sola alla fine; poi si somma su tutti i prodotti.
        import_penalty = (S * M_export).sum(axis=1)

        F = M_export @ Q_old - import_penalty

        # La penalita' non deve superare la fitness lorda: se accade, il
        # regime e' sbagliato (tipicamente alpha troppo grande) e tutto cio'
        # che segue (1/F) perde significato.
        # Con lo sfasamento corretto la penalita' e' vincolata: sottrae la sola
        # frazione importata dei contributi di ordine >=1 a Q, quindi
        # penalty_c < sum_p M^E_cp Q_p e F resta positiva. Una F<=0 segnala
        # un errore di scala, non un regime.
        if np.any(F <= 0):
            raise ValueError(
                f"iterazione {i}: {int((F <= 0).sum())} paesi con fitness non positiva. "
                "La penalita' da import supera la fitness lorda: controllare alpha, "
                "l'accoppiamento fra potenze di C e slice di Delta, e M_import."
            )

        # 2.2.2) Complessita' intrinseca Delta
        with np.errstate(divide="ignore", invalid="ignore"):
            Delta_new = 1.0 / (M_export.T @ (1.0 / F_old))
        Delta_new[~np.isfinite(Delta_new)] = 0.0

        mean_Delta_new = np.mean(Delta_new)
        Delta_new = Delta_new / mean_Delta_new

        # 2.2.3) Complessita' totale Q
        CQ = alpha * (C @ Q_old)
        Q  = CQ + Delta_new

        # 2.3) Normalizzazione
        mean_Q = np.mean(Q)
        Delta_out = Delta_new / mean_Q
        CQ        = CQ / mean_Q
        Q         = Q / mean_Q
        F         = F / np.mean(F)

        # 2.4) Procedura Delta

        # shifto (lasciando vuota la posizione 0) e normalizzo
        Delta[1:] = Delta[:-1] / mean_Q

        # fillo la posizione 0 con il valore di delta_new
        Delta[0]  = Delta_out

        # *.a) CONVERGENZA
        if track_convergence:
            history_convergence.append(
                (np.mean(np.abs(F - F_old)), np.mean(np.abs(Q - Q_old)))
            )

        # *.b) DELTA (storico per output, indipendente dal tensore interno)
        if save_delta:
            for d in delta_hist:
                d /= mean_Q
            delta_hist.append(Delta[0].copy())

        # 3) Convergenza (criterio relativo)
        if (np.max(np.abs(Q - Q_old) / np.maximum(Q, 1e-300)) < tol
                and np.max(np.abs(F - F_old) / np.maximum(F, 1e-300)) < tol):
            break

    fitness              = pd.Series(F,         index=Mcp_export.index,   name="Fitness")
    import_penalty       = pd.Series(import_penalty, index=Mcp_export.index,   name="Import penalty")
    complexity           = pd.Series(Q,         index=Mcp_export.columns, name="Complexity")
    input_complexity     = pd.Series(CQ,        index=Mcp_export.columns, name="Input Complexity")
    intrinsic_complexity = pd.Series(Delta_out, index=Mcp_export.columns, name="Intrinsic Complexity")

    # *) Risultati opzionali
    info_convergence = None
    delta_history = None

    if track_convergence:
        info_convergence = {"n_iter": i, "mean_Q": mean_Q,
                            "history": np.array(history_convergence)}
    if save_delta:
        delta_history = np.array(delta_hist)

    if track_convergence or save_delta:
        return (complexity, fitness, input_complexity, intrinsic_complexity, import_penalty,
                info_convergence, delta_history)

    return complexity, fitness, input_complexity, intrinsic_complexity, import_penalty


PARTI DA ANALIZZARE

In [ ]:
# non capisco bene quale sia il senso di rinominare con B che non c'entra nulla con la mia trattazione
# da capire inoltre cosa fanno le funzioni all'interno
    B = sp.csr_matrix(np.ascontiguousarray(C.T))

In [ ]:
K = tensor_window

# Delta non è un vettore ma un tensore, non capisco questa notazione con un solo indice tra parentesi
R = Delta[K] * M_import                      

In [ ]:
# in che senso ci sono 3 argomenti su range?
for m in range(K - 1, 0, -1):
    R = R @ B                                
    R = Delta[m] * M_import + alpha * R      
    S = alpha * np.asarray(R @ B)                


In [ ]:
# 2.2.1) Fitness F

        # 2.2.1.a) Calcolo import penalty (schema di Horner)
        
       
        K = tensor_window

        # FILTRO 1 (import): applicato a ogni ordine separatamente
        R = Delta[K] * M_import                      # X_K
        for m in range(K - 1, 0, -1):
            R = R @ B                                # un passo a valle nella rete
            R = Delta[m] * M_import + alpha * R      # aggiungo l'import di ordine m
        S = alpha * np.asarray(R @ B)                # (n_c, n_p)

        # FILTRO 2 (export): identico per tutti gli ordini, quindi si applica
        # una volta sola alla fine; poi si somma su tutti i prodotti.
        import_penalty = (S * M_export).sum(axis=1)

        F = M_export @ Q_old - import_penalty

        # La penalita' non deve superare la fitness lorda
        
        if np.any(F <= 0):
            raise ValueError(
                f"iterazione {i}: {int((F <= 0).sum())} paesi con fitness non positiva. "
                "La penalita' da import supera la fitness lorda: controllare alpha, "
                "l'accoppiamento fra potenze di C e slice di Delta, e M_import."
            )
